we are going to use the reference geolocation data set, i laready check this out here are some the issues
some of the city is not normalized some has different spelllings and we have extra columns that we dont need for validations 

In [1]:
import pandas as pd

geo_ref = pd.read_excel('/Users/john/Documents/data_clean/csv-files/ceps.xlsx')
customer_info = pd.read_csv('../csv-files/olist_customers_dataset.csv')

# now we have crated a variable to call our data frames lets proceed to cleaning our geo_reference table

In [3]:
geo_rows = len(geo_ref)
geo_ref.info()

<class 'pandas.DataFrame'>
RangeIndex: 6614 entries, 0 to 6613
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID                         6614 non-null   int64  
 1   UF                         6614 non-null   str    
 2   REGIAO                     6614 non-null   str    
 3   LOCALIDADE                 6614 non-null   str    
 4   LOCALIDADE_SEM_ACENTOS     6614 non-null   str    
 5   FAIXA_DE_CEP               6614 non-null   str    
 6   CEP_INICIAL                6614 non-null   int64  
 7   CEP_FINAL                  6614 non-null   int64  
 8   SITUACAO                   6614 non-null   str    
 9   TIPO_DE_FAIXA              6614 non-null   str    
 10  LATITUDE                   6520 non-null   str    
 11  LONGITUDE                  6520 non-null   str    
 12  COD_GEOGRAFICO_SUBDIVISAO  6520 non-null   float64
 13  COD_GEOGRAFICO_DISTRITO    6520 non-null   float64
 14  COD

In [4]:
# drop columns that are insignificant
geo_ref = geo_ref.drop(columns=['ID', 'REGIAO', 'LOCALIDADE', 'FAIXA_DE_CEP',
                                'SITUACAO', 'TIPO_DE_FAIXA', 'COD_GEOGRAFICO_SUBDIVISAO', 
                                'COD_GEOGRAFICO_DISTRITO', 'COD_IBGE', 'MICRORREGIAO', 
                                'MESORREGIAO', 'CATEGORIA', 'ALTITUDE', 'LOCALIZACAO',
                                ])

In [5]:
# rename columnns
geo_ref = geo_ref.rename(columns={
    'LOCALIDADE_SEM_ACENTOS' : 'ref_city',
    'CEP_INICIAL' : 'ref_prefix_i',
    'CEP_FINAL' : 'ref_prefix_f',
    'LATITUDE' : 'ref_latitude',
    'LONGITUDE' : 'ref_longiitude',
    'LOCALIZACAO_SEM_ACENTOS' : 'ref_state',
    })

In [6]:
# we are going to clean the ref_state and ref_city column because we still have some delimeters and unicode text
import unicodedata
def remove_accents(text):
    if pd.isna(text):
        return text
    # Normalize and strip the accent markers
    return "".join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')


geo_ref['ref_state'] = geo_ref['ref_state'].str.split(',').str[1].str.strip().str.lower().apply(remove_accents)
geo_ref['ref_city'] = geo_ref['ref_city'].str.strip().str.lower().apply(remove_accents).str.replace("'", '', regex=False)
geo_ref['UF'] = geo_ref['UF'].str.lower()
print(geo_ref.head(5))

   UF      ref_city  ref_prefix_i  ref_prefix_f  ref_latitude ref_longiitude  \
0  ac    acrelandia      69945000      69949999  -10,07379389   -67,05231658   
1  ac  assis brasil      69935000      69939999  -10,94286568   -69,56345917   
2  ac     brasileia      69932000      69933999  -11,01641102   -68,74794343   
3  ac        bujari      69926000      69926999  -9,820019627   -67,95185928   
4  ac      capixaba      69931000      69931999  -10,56976771   -67,67350797   

  ref_state  
0      acre  
1      acre  
2      acre  
3      acre  
4      acre  


In [7]:
# checking duplicates on our geo reference dataframe sort them so we can easily spot duplicates 
duplicate_ranges_df = geo_ref[geo_ref.duplicated(keep=False)]
duplicate_ranges_df = duplicate_ranges_df.sort_values(by=['ref_city', 'ref_prefix_i', 'ref_prefix_f'])
display(duplicate_ranges_df.head())


,UF,ref_city,ref_prefix_i,ref_prefix_f,ref_latitude,ref_longiitude,ref_state
4288,sp,adamantina,17800000,17809999,"-21,68831148","-51,07336475",sao paulo
4289,sp,adamantina,17800000,17809999,"-21,68831148","-51,07336475",sao paulo
4290,sp,adolfo,15230000,15239999,"-21,23272978","-49,64972143",sao paulo
4291,sp,adolfo,15230000,15239999,"-21,23272978","-49,64972143",sao paulo
4292,sp,aguai,13860000,13869999,"-22,059684","-46,97969311",sao paulo


In [8]:
# clean up duplicates and put some comparison to check original number of rows and cleaned rows
geo_ref_old = geo_ref.copy()
geo_ref = geo_ref.drop_duplicates()

print(f"Original rows: {len(geo_ref_old)}")
print(f"Cleaned rows remaining: {len(geo_ref)}")


Original rows: 6614
Cleaned rows remaining: 6153


In [9]:
#extracting the prefix and padding to meet the 5 digit standard for CEPs Brazil
extract_zip_i = geo_ref['ref_prefix_i'].astype(str).str[:-3] #extract
extract_zip_f = geo_ref['ref_prefix_f'].astype(str).str[:-3]

padded_i = (extract_zip_i.str.len() <5).sum()
padded_f = (extract_zip_f.str.len() <5).sum()

geo_ref['ref_prefix_i'] = extract_zip_i.str.zfill(5) #padd
geo_ref['ref_prefix_f'] = extract_zip_f.str.zfill(5)

print("i_prefix padded:", padded_i)
print("f_prefix padded:", padded_f)
print(geo_ref.head())


i_prefix padded: 44
f_prefix padded: 44
   UF      ref_city ref_prefix_i ref_prefix_f  ref_latitude ref_longiitude  \
0  ac    acrelandia        69945        69949  -10,07379389   -67,05231658   
1  ac  assis brasil        69935        69939  -10,94286568   -69,56345917   
2  ac     brasileia        69932        69933  -11,01641102   -68,74794343   
3  ac        bujari        69926        69926  -9,820019627   -67,95185928   
4  ac      capixaba        69931        69931  -10,56976771   -67,67350797   

  ref_state  
0      acre  
1      acre  
2      acre  
3      acre  
4      acre  


In [10]:
# rename column name for our customer info data frame
customer_info = customer_info.rename(columns={
    'customer_id' : 'cust_id',
    'customer_unique_id' : 'cust_unique_id',
    'customer_zip_code_prefix' : 'cust_zip_code_prefix',
    'customer_city' : 'cust_city',
    'customer_state' : 'cust_state',
})

print(customer_info.head())

                            cust_id                    cust_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   cust_zip_code_prefix              cust_city cust_state  
0                 14409                 franca         SP  
1                  9790  sao bernardo do campo         SP  
2                  1151              sao paulo         SP  
3                  8775        mogi das cruzes         SP  
4                 13056               campinas         SP  


In [115]:
# 1. Get the lengths of all values in the column
lengths_unique_id = customer_info['cust_unique_id'].str.len()
lengths_id = customer_info['cust_id'].str.len()


# 2. Find the most common length (the mode)
standard_length_unique_id = lengths_unique_id.mode()[0]
standard_length_id = lengths_id.mode()[0]

print(f"cust_unique_id common length is: {standard_length_unique_id} characters\n")
print(f"cust_id common length is: {standard_length_id} characters\n")

# 3. Find rows that do NOT match this standard length
invalid_length_unique_id = lengths_unique_id != standard_length_unique_id
invalid_length_id = lengths_id != standard_length_id

invalid_rows_unique_id = customer_info[invalid_length_unique_id]
invalid_rows_cust_id = customer_info[invalid_length_id]

# 4. Display the results
print(f"Found {len(invalid_rows_unique_id)} rows that don't match the standard length for cust_unique_id.")
print(f"Found {len(invalid_rows_cust_id)} rows that don't match the standard length for cust_id.")


cust_unique_id common length is: 32 characters

cust_id common length is: 32 characters

Found 0 rows that don't match the standard length for cust_unique_id.
Found 0 rows that don't match the standard length for cust_id.


In [11]:
# Clean the city column
# Remove Brazilian accent marks (e.g., 'são paulo' becomes 'sao paulo')
# This requires the 'unicodedata' library (built into Python)
import unicodedata
def remove_accents(text):
    if pd.isna(text):
        return text
    # Normalize and strip the accent markers
    return "".join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

# Convert to lowercase, strip leading/trailing spaces and unicode/accent text
customer_info['cust_city'] = customer_info['cust_city'].str.lower().str.strip().apply(remove_accents).str.replace('-', ' ', regex=False).str.replace("'", '', regex=False)
customer_info['cust_state'] = customer_info['cust_state'].str.lower().str.strip().apply(remove_accents).str.replace('-', ' ', regex=False)

# 2. Check your new duplicate count (Optional)
# This will likely be lower now because variations are merged
print("Duplicates Removed:", customer_info.duplicated().sum())

# 3. Drop the exact duplicates across the whole row
customer_info = customer_info.drop_duplicates()
print("Final row count after clean text drop:", len(customer_info))


Duplicates Removed: 0
Final row count after clean text drop: 99441


In [12]:
# Count rows that need padding (less than 5 digits)
customer_padded_count = (customer_info['cust_zip_code_prefix'].astype(str).str.len() < 5).sum()

print(f"Customer rows padded:    {customer_padded_count}")

# Then apply the padding
customer_info['cust_zip_code_prefix'] = customer_info['cust_zip_code_prefix'].astype(str).str.zfill(5)

Customer rows padded:    23995


In [13]:
#bloated approach

# 1. Keep track of your original row order by adding a temporary ID
customer_info['temp_row_id'] = range(len(customer_info))

# 2. Merge on city name (this will create duplicate rows for cities with multiple ranges)
exploded_df = customer_info.merge(
    geo_ref[['ref_city', 'UF', 'ref_prefix_i', 'ref_prefix_f']], 
    left_on=['cust_city', 'cust_state'], 
    right_on=['ref_city', 'UF'],
    how='left'
)

# 3. Fill NaN values to safely convert to integers without crashing
ref_i_safe = exploded_df['ref_prefix_i'].fillna(-1).astype(int)
ref_f_safe = exploded_df['ref_prefix_f'].fillna(-1).astype(int)
cust_zip_safe = exploded_df['cust_zip_code_prefix'].fillna(-1).astype(int)

# 4. Check if the zip code matches THIS specific range row
exploded_df['is_valid_range'] = (cust_zip_safe >= ref_i_safe) & (cust_zip_safe <= ref_f_safe) & (exploded_df['ref_city'].notna())

# 5. Collapse the rows back down to the original shape using groupby!
# .any() means: "If the customer matched AT LEAST ONE of their city's ranges, mark it True"
validation_map = exploded_df.groupby('temp_row_id')['is_valid_range'].any()

# 6. Map the final True/False validation back to your clean customer dataset
customer_info['is_zip_valid'] = customer_info['temp_row_id'].map(validation_map)

# 7. Drop the temporary ID column to keep things clean
customer_info.drop(columns=['temp_row_id'], inplace=True)

# 8. Filter your final results perfectly
valid_matches = customer_info[customer_info['is_zip_valid'] == True]
mismatches = customer_info[customer_info['is_zip_valid'] == False]

print("Valid geometric matches (no row duplicates):", len(valid_matches))
print("Mismatched or out-of-range rows:", len(mismatches))
print("Total rows matches original shape?", len(customer_info) == (len(valid_matches) + len(mismatches)))
display(mismatches.head())

Valid geometric matches (no row duplicates): 98006
Mismatched or out-of-range rows: 1435
Total rows matches original shape? True


,cust_id,cust_unique_id,cust_zip_code_prefix,cust_city,cust_state,is_zip_valid
54,8247b5583327ab8be19f96e1fb82f77b,d85547cd859833520b311b4458a14c1c,23970,parati,rj,False
142,d387341bbce5ab96e3647bb0e8d0b55a,91e6fd64694fa9a828270d442ba88e03,06835,embu,sp,False
215,e48f50d453ada5bceb41e239a2dc2065,9d85c39a38a86ed13978803d7560c612,75570,bom jesus,go,False
227,d6b41b191f1f643ff583898156e049cf,3a62a4803c77457a96c996a9db36a662,28375,varre sai,rj,False
261,4308615296cf4ca17defd8b0d59287d8,a99021699fda65c2b84a3fb596a4c32b,76974,espigao do oeste,ro,False


In [14]:


# --- PRE-REQUISITE CHECK ---
# Ensure customer_info has 'temp_row_id' to align with our range maps
customer_info['temp_row_id'] = range(len(customer_info))

# --- STEP 1: BUILD THE RANGE-AWARE MAPS ---
# Match customer data to ALL possible geo_ref ranges based purely on the zip code prefix
zip_only_match = customer_info.merge(
    geo_ref[['ref_city', 'UF', 'ref_prefix_i', 'ref_prefix_f']],
    left_on='cust_zip_code_prefix',   # Your customer zip column
    right_on='ref_prefix_i',          # FIXED: Explicitly link it to the reference start zip
    how='inner'                       # Inner join keeps only zip codes that exist in reference ranges
)

# Convert to integers for exact range logic math
cust_zip_int = zip_only_match['cust_zip_code_prefix'].fillna(-1).astype(int)
ref_i_int = zip_only_match['ref_prefix_i'].fillna(-1).astype(int)
ref_f_int = zip_only_match['ref_prefix_f'].fillna(-1).astype(int)

# Filter for rows where the customer zip falls perfectly inside a valid window
valid_zip_mask = (cust_zip_int >= ref_i_int) & (cust_zip_int <= ref_f_int)
zip_only_match = zip_only_match[valid_zip_mask]

# Blueprint A: Maps Row ID -> True City & State (For when the text was wrong/blank)
true_geo_map = zip_only_match.drop_duplicates(subset=['temp_row_id']).set_index('temp_row_id')[['ref_city', 'UF']].to_dict('index')

# Blueprint B: Maps (City, State) -> Default Start Zip (For when the zip code was out of range)
city_to_zip_map = geo_ref.drop_duplicates(subset=['ref_city', 'UF']).set_index(['ref_city', 'UF'])['ref_prefix_i'].to_dict()


# Convert to integers for exact range logic math
cust_zip_int = zip_only_match['cust_zip_code_prefix'].fillna(-1).astype(int)
ref_i_int = zip_only_match['ref_prefix_i'].fillna(-1).astype(int)
ref_f_int = zip_only_match['ref_prefix_f'].fillna(-1).astype(int)

# Filter for rows where the customer zip falls perfectly inside a valid window
valid_zip_mask = (cust_zip_int >= ref_i_int) & (cust_zip_int <= ref_f_int)
zip_only_match = zip_only_match[valid_zip_mask]

# Blueprint A: Maps Row ID -> True City & State (For when the text was wrong/blank)
true_geo_map = zip_only_match.drop_duplicates(subset=['temp_row_id']).set_index('temp_row_id')[['ref_city', 'UF']].to_dict('index')

# Blueprint B: Maps (City, State) -> Default Start Zip (For when the zip code was out of range)
city_to_zip_map = geo_ref.drop_duplicates(subset=['ref_city', 'UF']).set_index(['ref_city', 'UF'])['ref_prefix_i'].to_dict()


# --- STEP 2: DEFINE THE COUNTERS & HEALER ---
# Track validation groups
fixed_by_zip_count = 0
fixed_by_city_count = 0
still_broken_count = 0

def auto_heal_with_metrics(row):
    global fixed_by_zip_count, fixed_by_city_count, still_broken_count
    
    row_id = row['temp_row_id']
    city = row['cust_city']
    state = row['cust_state']
    
    # FIX TYPE 1: Zip is valid. Use the range to fix broken/blank City and State text
    if row_id in true_geo_map:
        row['cust_city'] = true_geo_map[row_id]['ref_city']
        row['cust_state'] = true_geo_map[row_id]['UF']
        row['is_zip_valid'] = True
        fixed_by_zip_count += 1
        
    # FIX TYPE 2: City/State text is valid. Assign the city's base zip prefix
    elif (city, state) in city_to_zip_map:
        row['cust_zip_code_prefix'] = city_to_zip_map[(city, state)]
        row['is_zip_valid'] = True
        fixed_by_city_count += 1
        
    # FAILURE CASE: Both the text and the zip prefix are corrupted or unmapped
    else:
        still_broken_count += 1
        
    return row

# FIX: Force-inject the tracking IDs back into your validation subsets
mismatches['temp_row_id'] = mismatches.index
valid_matches['temp_row_id'] = valid_matches.index


# --- STEP 3: RUN THE HEALER ONLY ON MISMATCHES ---
print("--- STARTING AUTO-HEALER PIPELINE ---")
healed_mismatches = mismatches.copy().apply(auto_heal_with_metrics, axis=1)


# --- STEP 4: COMBINE AND GENERATE REPORT ---
# Recombine original valid rows with your newly healed rows
final_clean_customers = pd.concat([valid_matches, healed_mismatches]).drop(columns=['temp_row_id']).reset_index(drop=True)

# Math for the tracking report
total_initial_mismatches = len(mismatches)
total_healed = fixed_by_zip_count + fixed_by_city_count
recovery_rate = (total_healed / total_initial_mismatches) * 100

print("\n==================================================")
print("             DATA VALIDATION REPORT               ")
print("==================================================")
print(f"Initial Mismatched Rows:          {total_initial_mismatches}")
print(f" -> Fixed by Zip Code Range:      {fixed_by_zip_count}")
print(f" -> Fixed by City/State Name:     {fixed_by_city_count}")
print(f" -> Still Needs Manual Cleaning:  {still_broken_count}")
print("--------------------------------------------------")
print(f"Total Rows Auto-Healed:           {total_healed}")
print(f"Pipeline Automation Success Rate: {recovery_rate:.2f}%")
print(f"Final Clean Customer Row Count:   {len(final_clean_customers)}")
print("==================================================")


--- STARTING AUTO-HEALER PIPELINE ---

             DATA VALIDATION REPORT               
Initial Mismatched Rows:          1435
 -> Fixed by Zip Code Range:      306
 -> Fixed by City/State Name:     660
 -> Still Needs Manual Cleaning:  469
--------------------------------------------------
Total Rows Auto-Healed:           966
Pipeline Automation Success Rate: 67.32%
Final Clean Customer Row Count:   99441


In [16]:
still_broken_df = final_clean_customers[final_clean_customers['is_zip_valid'] == False]
table = still_broken_df[['cust_zip_code_prefix', 'cust_city', 'cust_state' ]].value_counts()
high_frequency_problems = table[table > 10]
single_count_noise = table[table == 1]

print(f"Total distinct problem cities: {len(table)}")
print(f"Cities with MULTIPLE errors (Worth looking at): {len(high_frequency_problems)}")
print(f"Cities with only ONE error (Safe to ignore/drop): {len(single_count_noise)}")
print(high_frequency_problems)


Total distinct problem cities: 265
Cities with MULTIPLE errors (Worth looking at): 3
Cities with only ONE error (Safe to ignore/drop): 181
cust_zip_code_prefix  cust_city          cust_state
14110                 bonfim paulista    sp            18
28880                 barra de sao joao  rj            13
28110                 goitacazes         rj            11
Name: count, dtype: int64


In [17]:
customer_info = final_clean_customers.copy()

In [18]:


# 1. Your dictionary of manual corrections
zip_corrections = {
    ('14110', 'bonfim paulista', 'sp'): "ribeirao preto",
    ('28880', 'barra de sao joao', 'rj'): "casimiro de abreu",
    ('28110', 'goitacazes', 'rj'): "campos dos goytacazes"
}

# 2. Build the lookup table (using the loop you prefer)
formatted_data = []
for (old_zip, city, state), new_city in zip_corrections.items():
    row = (old_zip, city, state, new_city)
    formatted_data.append(row)

lookup_df = pd.DataFrame(
    formatted_data, 
    columns=['cust_zip_code_prefix', 'cust_city', 'cust_state', 'new_city']
)

# 3. Match the lookup table with your 100k rows (Left Join)
customer_info = customer_info.merge(
    lookup_df, 
    on=['cust_zip_code_prefix', 'cust_city', 'cust_state'], 
    how='left'
)

# 4. Overwrite old city names with new ones where a match was found
customer_info['cust_city'] = customer_info['new_city'].fillna(customer_info['cust_city'])

# 5. Drop the temporary column used for matching
customer_info.drop(columns=['new_city'], inplace=True)


In [19]:
# 1. Filter your dataset for the specific ZIP codes you targeted
validation_check = customer_info[
    customer_info['cust_zip_code_prefix'].isin(['14110', '28860', '28110'])
]

# 2. Count how many times the old incorrect names still show up
remaining_errors = validation_check[
    validation_check['cust_city'].isin(['bonfim paulita', 'barra de sao joao', 'goitacazes'])
].shape[0]

# 3. Print the results
print(f"--- Verification Results ---")
print(f"Remaining rows with old incorrect names: {remaining_errors}")

if remaining_errors == 0:
    print("Success! All target names were completely cleaned across your 100k rows.")
else:
    print("Warning: Some old names are still there. Check for capitalization or spelling mismatches!")

# 4. View a quick preview of your updated data to see the new names
print("\nPreview of updated rows:")
print(validation_check[['cust_zip_code_prefix', 'cust_city', 'cust_state']].drop_duplicates())


--- Verification Results ---
Remaining rows with old incorrect names: 0
Success! All target names were completely cleaned across your 100k rows.

Preview of updated rows:
      cust_zip_code_prefix              cust_city cust_state
1553                 28860      casimiro de abreu         rj
98026                14110         ribeirao preto         sp
98097                28110  campos dos goytacazes         rj


In [20]:
# =====================================================================
# FINAL QUALITY ASSURANCE & VERIFICATION SCRIPT (EXPLOSION FIX)
# =====================================================================

# 1. Merge customer data with the reference ranges on city and state
# (This creates temporary row duplicates for multi-range cities)
final_check_df = customer_info.merge(
    geo_ref[['ref_city', 'UF', 'ref_prefix_i', 'ref_prefix_f']], 
    left_on=['cust_city', 'cust_state'], 
    right_on=['ref_city', 'UF'],
    how='left'
)

# 2. Safely cast prefixes to integers to handle numeric validation
ref_i_final = final_check_df['ref_prefix_i'].fillna(-1).astype(int)
ref_f_final = final_check_df['ref_prefix_f'].fillna(-1).astype(int)
cust_zip_final = final_check_df['cust_zip_code_prefix'].fillna(-1).astype(int)

# 3. Check if the customer's zip code fits within ANY of the city's matching ranges
final_check_df['is_geometrically_valid'] = (
    (cust_zip_final >= ref_i_final) & 
    (cust_zip_final <= ref_f_final) & 
    (final_check_df['ref_city'].notna())
)

# 4. FIXED: Collapse the exploded table back down using your unique 'cust_id' column
# This eliminates the 18,275 duplicate rows entirely!
final_validation_results = final_check_df.groupby('cust_id')['is_geometrically_valid'].any()

# 5. Extract exact, un-duplicated counts matching your true dataset shape
total_rows = len(customer_info)
fully_verified_rows = final_validation_results.sum()
remaining_mismatches = total_rows - fully_verified_rows
final_accuracy_percentage = (fully_verified_rows / total_rows) * 100

# 6. PRINT TRUE FINAL REPORT
print("==================================================")
print("          FINAL GEOGRAPHIC AUDIT REPORT           ")
print("==================================================")
print(f"Total Rows Evaluated:            {total_rows:,}")
print(f"Successfully Verified Rows:      {fully_verified_rows:,}")
print(f"Remaining Mismatched Rows:       {remaining_mismatches:,}")
print("--------------------------------------------------")
print(f"Final Dataset Data Integrity:    {final_accuracy_percentage:.2f}%")
print("==================================================")

if remaining_mismatches == 0:
    print("🏆 SUCCESS! Every single row perfectly aligns with its geometric range.")
else:
    print(f"⚠️ Note: {remaining_mismatches} rows did not match. This matches your remaining single-count noise.")


          FINAL GEOGRAPHIC AUDIT REPORT           
Total Rows Evaluated:            99,441
Successfully Verified Rows:      99,014
Remaining Mismatched Rows:       427
--------------------------------------------------
Final Dataset Data Integrity:    99.57%
⚠️ Note: 427 rows did not match. This matches your remaining single-count noise.


In [21]:
customer_info.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   cust_id               99441 non-null  str  
 1   cust_unique_id        99441 non-null  str  
 2   cust_zip_code_prefix  99441 non-null  str  
 3   cust_city             99441 non-null  str  
 4   cust_state            99441 non-null  str  
 5   is_zip_valid          99441 non-null  bool 
dtypes: bool(1), str(5)
memory usage: 3.9 MB


In [25]:
customer_info = customer_info.drop(columns = 'is_zip_valid')

In [122]:
display(geo_ref[geo_ref['ref_city'] == 'ribeirao preto'])
display(customer_info[customer_info['cust_city'] == 'ribeirao preto'])

,UF,ref_city,ref_prefix_i,ref_prefix_f,ref_latitude,ref_longiitude,ref_state
6000,sp,ribeirao preto,14000,14114,"-21,1848345","-47,80547592",sao paulo
6001,sp,ribeirao preto,14000,14109,"-21,1848345","-47,80547592",sao paulo


,cust_id,cust_unique_id,cust_zip_code_prefix,cust_city,cust_state,is_zip_valid,temp_row_id
44,c532a74a3ebf1bacce2e2bcce3783317,91ec50a00ae74d0a229d2efdf4344e1e,14026,ribeirao preto,sp,True,44
406,11ba86af0ea78e6db913284c2e9902a8,273fff5a1d5e0ffc35f76b3fe3fb2516,14050,ribeirao preto,sp,True,406
420,51ce96915c12887208b633982c4d8435,fb1756887ae59c1b0efe8d0544d94815,14051,ribeirao preto,sp,True,420
510,00bb2dd15521a8a428bb530b4cabb403,0cbd5dfde4c06bbe2960d2c0c95e5e7d,14055,ribeirao preto,sp,True,510
881,08be5d1f213ea37487f2698ca4b2be07,3eddcc05abf2cf434aae2a9662597353,14015,ribeirao preto,sp,True,881
...,...,...,...,...,...,...,...
98843,b5dcdf41ce407279527c68ee494abecc,6f33a8b7e82b5685bab383f1aa5ab8cb,14093,ribeirao preto,sp,True,98843
99244,98b8e49c0cf885f1da80403205b422a6,7218da49ef6bca3a8d862b83e6112cf5,14066,ribeirao preto,sp,True,99244
99280,9ce70b8dafe76524f3d5e5e68cbf29f5,6edd9e4a22b7ab02db2a87f13bfd2d2a,14093,ribeirao preto,sp,True,99280
99303,6df3e1d2f32e019f4bad7640293ba4d3,c1a9251508def21d0924f9eb2ed8d3bc,14095,ribeirao preto,sp,True,99303


In [123]:
#ref_city checker
city_exists = geo_ref['ref_city'].str.contains('lagoa santa', case=False, na=False).any()
print(city_exists)

True


In [124]:
#ref_state checker
state_exists = geo_ref['UF'].str.contains('mg', case=False, na=False).any()
print(state_exists)

True


In [125]:
city_exists = customer_info['cust_city'].str.contains('ribeirao preto', case=False, na=False).any()
print(city_exists)

True


In [28]:
customer_info.head()


,cust_id,cust_unique_id,cust_zip_code_prefix,cust_city,cust_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,sp
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,sp
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,sp
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,sp
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,sp


In [29]:
customer_info.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   cust_id               99441 non-null  str  
 1   cust_unique_id        99441 non-null  str  
 2   cust_zip_code_prefix  99441 non-null  str  
 3   cust_city             99441 non-null  str  
 4   cust_state            99441 non-null  str  
dtypes: str(5)
memory usage: 3.8 MB


In [30]:
customer_info.to_csv('../df_cleaned/customer_info_cln.csv')